# Jablje — Soil Property Comparison (Daily)

Compares the effect of two different `ksat` / `cra` / `crb` value sets for the
B1+B2 60-110 cm layer on daily AquaCrop outputs (soil water content, water
fluxes, canopy cover, biomass and yield). All other soil layers, crop
parameters and management stay identical between the two runs.

- **Set A** — current measured/GUI-derived values (ksat=3.4, cra=-0.567836, crb=-2.994180)
- **Set B** — alternate values (ksat=8600, cra=-0.911700, crb=1.646146)


In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import dataclasses
from datetime import date

from aquacrop import AquaCrop, Crop, Soil, Weather

from aquacrop_slovenia import config
from aquacrop_slovenia.reading_data import get_co2_for_aquacrop, get_station_weather
from aquacrop_slovenia.parameters_running_jablje import jablje_soil_layers, jablje_curve_number, \
    jablje_readily_evaporable_water, jablje_maize_params, jablje_management, jablje_initial_cond, \
    jablje_groundwater
import locale
locale.setlocale(locale.LC_ALL, "sl")

## Two soil variants

Both soils share every layer except the B1+B2 60-110 cm layer (index 2), where `ksat`, `cra` and `crb` differ between Set A and Set B.

In [ ]:
LOCATION = "jablje"
SOIL_VARIANTS = {
    "Set A (ksat=3.4)": {"ksat": 3.4, "cra": -0.567836, "crb": -2.994180},
    "Set B (ksat=8600)": {"ksat": 8600, "cra": -0.911700, "crb": 1.646146},
}
VARIED_LAYER_INDEX = 2  # B1+B2 60-110

def build_soil_layers(overrides):
    layers = list(jablje_soil_layers)
    layers[VARIED_LAYER_INDEX] = dataclasses.replace(layers[VARIED_LAYER_INDEX], **overrides)
    return layers

jablje_soil_variants = {
    name: Soil(
        name=f"{LOCATION} soil - {name}",
        description=f"{LOCATION} silt loam soil, B1+B2 60-110 layer varied ({name})",
        soil_layers=build_soil_layers(overrides),
        curve_number=jablje_curve_number,
        readily_evaporable_water=jablje_readily_evaporable_water,
    )
    for name, overrides in SOIL_VARIANTS.items()
}
jablje_soil_variants

In [ ]:
# Set up working directories for outputs, one per soil variant
output_dirs = {
    name: config.MODEL_RUNNING_DIR / "soil_comparison" / slug
    for name, slug in zip(SOIL_VARIANTS, ["set_a", "set_b"])
}
output_dirs

In [ ]:
year = 2022
SPINUP_YEARS = 3

simulation_periods = [
    {
        "start_date": date(y, 1, 1),
        "end_date": date(y, 12, 31),
        "planting_date": date(y, 5, 3),
        "is_seeding_year": True,
    }
    for y in range(year - SPINUP_YEARS, year + 1)
]

In [ ]:
jablje_temperatures, jablje_eto, jablje_rain = get_station_weather(8)  # Jablje uses ARSO meteo station 8 data
historical_co2 = get_co2_for_aquacrop("historical")

jablje_weather = Weather(
    location=LOCATION,
    temperatures=jablje_temperatures,
    eto_values=jablje_eto,
    rainfall_values=jablje_rain,
    record_type=1,
    first_day=1,
    first_month=1,
    first_year=1993,
    co2_records=historical_co2
)

In [ ]:
jablje_maize = Crop(
    name=f"{LOCATION} maize",
    description=f"{LOCATION} maize uncalibrated",
    params=jablje_maize_params
)

## Run both simulations

In [ ]:
results_by_variant = {}
for name, soil in jablje_soil_variants.items():
    simulation = AquaCrop(
        simulation_periods=simulation_periods,
        crop=jablje_maize,
        soil=soil,
        management=jablje_management,
        initial_conditions=jablje_initial_cond,
        climate=jablje_weather,
        ground_water=jablje_groundwater,
        working_dir=output_dirs[name],
        need_daily_output=True,
        need_seasonal_output=True,
        need_harvest_output=False,
        need_evaluation_output=False
    )
    results = simulation.run()
    results_by_variant[name] = results
    print(f"Finished: {name}")

In [ ]:
seasonal_by_variant = {name: r["season"] for name, r in results_by_variant.items()}
for name, seasonal in seasonal_by_variant.items():
    print(name)
    display(seasonal)

## Load daily output for the comparison year

In [ ]:
import pandas as pd
from aquacrop.output import OutputReader

day_by_variant = {}
for name, output_dir in output_dirs.items():
    reader = OutputReader(output_dir=str(output_dir / "OUTP"))
    reader.scan_directory()
    day_df = reader.get_day_data(run_number=SPINUP_YEARS + 1)
    day_df["date"] = pd.to_datetime(dict(year=day_df["Year"], month=day_df["Month"], day=day_df["Day"]))
    day_by_variant[name] = day_df

def to_num(df, col):
    s = pd.to_numeric(df[col], errors="coerce")
    return s.where(s != -9)  # replace -9 sentinel (outside season) with NaN

## Comparison plots

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

VARIANT_COLORS = {name: color for name, color in zip(SOIL_VARIANTS, ["#1f77b4", "#d62728"])}

fig, axes = plt.subplots(5, 1, figsize=(14, 17), sharex=True)

wc_total_col = next(c for c in next(iter(day_by_variant.values())).columns if c.startswith("WC("))
_ref_df = next(iter(day_by_variant.values()))

# Root zone water content
ax = axes[0]
for name, df in day_by_variant.items():
    ax.plot(df["date"], to_num(df, "Wr"), color=VARIANT_COLORS[name], label=name)
ax.plot(_ref_df["date"], to_num(_ref_df, "Wr(FC)"), color="green", linestyle="--", linewidth=1, label="Wr(FC)")
ax.plot(_ref_df["date"], to_num(_ref_df, "Wr(PWP)"), color="gray", linestyle="--", linewidth=1, label="Wr(PWP)")
ax.set_ylabel("Wr (mm)")
ax.set_title("Vsebnost vode v koreninski coni")
ax.legend(loc="upper left", fontsize=8)

# Total soil water content
ax = axes[1]
for name, df in day_by_variant.items():
    ax.plot(df["date"], to_num(df, wc_total_col), color=VARIANT_COLORS[name], label=name)
ax.set_ylabel(f"{wc_total_col} (mm)")
ax.set_title("Skupna vlaga tal")
ax.legend(loc="upper left", fontsize=8)

# Capillary rise
ax = axes[2]
for name, df in day_by_variant.items():
    ax.plot(df["date"], to_num(df, "CR"), color=VARIANT_COLORS[name], label=name)
ax.set_ylabel("CR (mm/dan)")
ax.set_title("Kapilarni dvig")
ax.legend(loc="upper left", fontsize=8)

# Canopy cover
ax = axes[3]
for name, df in day_by_variant.items():
    ax.plot(df["date"], to_num(df, "CC"), color=VARIANT_COLORS[name], label=name)
ax.set_ylabel("CC (%)")
ax.set_title("Pokrovnost rastlin")
ax.legend(loc="upper left", fontsize=8)

# Biomass & yield
ax = axes[4]
for name, df in day_by_variant.items():
    ax.plot(df["date"], to_num(df, "Biomass"), color=VARIANT_COLORS[name], linestyle="-", label=f"{name} biomasa")
    ax.plot(df["date"], to_num(df, "Y(dry)"), color=VARIANT_COLORS[name], linestyle="--", label=f"{name} pridelek")
ax.set_ylabel("t/ha")
ax.set_title("Biomasa in pridelek zrnja")
ax.legend(loc="upper left", fontsize=7, ncol=2)

for ax in axes:
    ax.xaxis_date()
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b"))
    ax.xaxis.set_major_locator(mdates.MonthLocator())
    ax.grid(True, alpha=0.3)
    ax.set_xlim(date(year, 1, 1), date(year, 12, 31))

plt.suptitle(f"Jablje {year} — Soil property comparison (B1+B2 60-110 layer)", fontsize=14)
plt.tight_layout()
plt.show()

## Soil water content by depth, side by side

In [ ]:
import pathlib
import numpy as np

wc_cols = ["WC01", "WC", "2", "WC_1", "3", "WC_2", "4", "WC_3", "5", "WC_4", "6", "WC_5"]

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True, sharey=True)

for ax, (name, output_dir) in zip(axes, output_dirs.items()):
    day_df = day_by_variant[name]

    day_file = next(pathlib.Path(output_dir / "OUTP").glob("*day.OUT"))
    with open(day_file) as f:
        units_line = f.readlines()[4]
    numeric_units = []
    for x in units_line.split():
        try:
            numeric_units.append(float(x))
        except ValueError:
            pass
    centers = np.array(numeric_units[:12])

    total_depth = float(wc_total_col[3:-1])
    bounds = np.concatenate([[0.0], (centers[:-1] + centers[1:]) / 2, [total_depth]])

    wc = day_df[wc_cols].values.astype(float)
    theta = wc / 100

    dates = mdates.date2num(day_df["date"].values)
    x_edges = np.concatenate([[dates[0] - 0.5], dates + 0.5])

    im = ax.pcolormesh(x_edges, bounds, theta.T, cmap="YlGnBu", shading="flat", vmin=0.1, vmax=0.5)
    ax.set_ylim(total_depth, 0)
    ax.xaxis_date()
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b"))
    ax.xaxis.set_major_locator(mdates.MonthLocator())
    ax.set_ylabel("Depth (m)")
    ax.set_title(name)
    plt.colorbar(im, ax=ax, label="θ (m³/m³)")

axes[-1].set_xlabel("Date")
plt.suptitle(f"Jablje {year} — Soil water content by depth", fontsize=14)
plt.tight_layout()
plt.show()

## Seasonal summary comparison

In [ ]:
summary_cols = ["Year1", "Rain", "ETo", "Irri", "Infilt", "CR", "Drain", "Biomass", "HI", "Y(dry)", "WPet"]
summary = pd.concat(
    {name: seasonal.loc[seasonal["Year1"] == year, [c for c in summary_cols if c in seasonal.columns]]
     for name, seasonal in seasonal_by_variant.items()},
    names=["Variant"]
).reset_index(level=0)
summary